# 02 — Inventario de cuadros
Una fila por página del PDF: tipo (cuadro, gráfica, vacía, sin etiqueta), identificador del
cuadro, título y muestra. Salida: `data/interim/catalogo_cuadros.csv`.

La etiqueta del cuadro no es uniforme ("Cuadro Nro.", "Cuadro No", "CUADRO Nº", "Cuadro 14.1.4")
y un cuadro puede ocupar varias páginas seguidas.

In [ ]:
%run -i modulos/comun.ipynb
import matplotlib.pyplot as plt

RE_CUADRO = re.compile(r"\b(CUADRO|Cuadro|cuadro)\s*(N(?:ro\.|ro|º|°|o\.|o))\s*([0-9]+(?:\.[0-9]+)*)")
# Sin abreviatura de número se exigen dos niveles (14.1.4) para no confundir con "cuadro 2023".
RE_CUADRO_SIN_N = re.compile(r"\b(CUADRO|Cuadro)\s+()([0-9]+(?:\.[0-9]+){1,})\b")
RE_GRAFICA = re.compile(r"\b(GR[ÁA]FICA|Gr[áa]fica)\s*(N(?:ro\.|ro|º|°|o\.|o))\s*([0-9]+(?:\.[0-9]+)*)")
RE_GESTION = re.compile(r"^\s*GESTI[ÓO]N\s*:?\s*\d{4}\s*$", re.I)
RE_NUM = re.compile(r"\d[\d.,]*%?")
UMBRAL_VACIA = 60

In [ ]:
def clasificar(pagina):
    # Devuelve (tipo, cuadro_id, variante de la etiqueta).
    if len(pagina.strip()) < UMBRAL_VACIA:
        return "vacia", None, None
    m = RE_CUADRO.search(pagina)
    if m:
        return "cuadro", m.group(3), m.group(1) + " " + m.group(2)
    m = RE_CUADRO_SIN_N.search(pagina)
    if m:
        return "cuadro", m.group(3), m.group(1) + " (sin Nro.)"
    g = RE_GRAFICA.search(pagina)
    if g:
        return "grafica", g.group(3), g.group(1) + " " + g.group(2)
    return "sin_etiqueta", None, None


def primera_linea_que_coincide(lineas, patron):
    for i in range(len(lineas)):
        if patron.search(lineas[i]):
            return i
    return None


def extraer_titulo(pagina):
    # Sección = línea anterior a la etiqueta; título = hasta 4 líneas siguientes, antes de la grilla.
    lineas = lineas_utiles(pagina)
    idx = primera_linea_que_coincide(lineas, RE_CUADRO)
    if idx is None:
        idx = primera_linea_que_coincide(lineas, RE_CUADRO_SIN_N)
    if idx is None:
        idx = primera_linea_que_coincide(lineas, RE_GRAFICA)
    if idx is None:
        return "", ""
    seccion = ""
    if idx > 0:
        seccion = " ".join(lineas[idx - 1].split())
    partes = []
    for linea in lineas[idx + 1: idx + 7]:
        texto = " ".join(linea.split())
        if RE_GESTION.match(texto):
            continue
        if len(RE_NUM.findall(texto)) >= 3:
            break
        partes.append(texto)
        if len(partes) >= 4:
            break
    return seccion, " / ".join(partes)


def lineas_de_datos(pagina, k=3):
    # Primeras k líneas con 3 o más números: parecen filas de datos.
    salida = []
    for linea in pagina.splitlines():
        texto = " ".join(linea.split())
        if not texto or es_ruido(linea):
            continue
        if len(RE_NUM.findall(texto)) >= 3:
            salida.append(texto[:150])
            if len(salida) >= k:
                break
    return salida

## Clasificación de las 774 páginas

In [ ]:
pags = paginas()
filas = []
n = 1
for pagina in pags:
    tipo, cuadro_id, variante = clasificar(pagina)
    seccion = ""
    titulo = ""
    if tipo == "cuadro" or tipo == "grafica":
        seccion, titulo = extraer_titulo(pagina)
    muestra = []
    if tipo == "cuadro":
        muestra = lineas_de_datos(pagina)
    if cuadro_id is None:
        cuadro_id = ""
    if variante is None:
        variante = ""
    filas.append({
        "pagina_pdf": n,
        "tipo": tipo,
        "cuadro_id": cuadro_id,
        "capitulo": cuadro_id.split(".")[0],
        "variante_etiqueta": variante,
        "seccion": seccion,
        "titulo": titulo,
        "n_lineas": len(lineas_utiles(pagina)),
        "n_lineas_datos": len(lineas_de_datos(pagina, k=10**6)),
        "primeras_lineas": " ¶ ".join(muestra),
    })
    n = n + 1
cat = pd.DataFrame(filas)
cat["tipo"].value_counts()

## Páginas consecutivas del mismo cuadro
Un cuadro que sigue en la página siguiente repite su identificador; se numeran las páginas del bloque.

In [ ]:
orden = []
total = []
rango = []
i = 0
while i < len(cat):
    cid = cat.at[i, "cuadro_id"]
    if cat.at[i, "tipo"] != "cuadro":
        orden.append(0)
        total.append(0)
        rango.append("")
        i = i + 1
        continue
    j = i
    while j + 1 < len(cat) and cat.at[j + 1, "tipo"] == "cuadro" and cat.at[j + 1, "cuadro_id"] == cid:
        j = j + 1
    n_pags = j - i + 1
    if n_pags == 1:
        etiqueta = str(cat.at[i, "pagina_pdf"])
    else:
        etiqueta = str(cat.at[i, "pagina_pdf"]) + "-" + str(cat.at[j, "pagina_pdf"])
    for k in range(n_pags):
        orden.append(k + 1)
        total.append(n_pags)
        rango.append(etiqueta)
    i = j + 1

cat["orden_pagina"] = orden
cat["paginas_del_cuadro"] = total
cat["rango_paginas"] = rango
cat = cat[["pagina_pdf", "cuadro_id", "titulo", "primeras_lineas", "tipo", "capitulo",
           "variante_etiqueta", "seccion", "orden_pagina", "paginas_del_cuadro", "rango_paginas",
           "n_lineas", "n_lineas_datos"]]

INTERIM.mkdir(parents=True, exist_ok=True)
cat.to_csv(INTERIM / "catalogo_cuadros.csv", index=False, encoding="utf-8")
print("catalogo_cuadros.csv:", len(cat), "filas")

## Resumen

In [ ]:
cuadros = cat[cat["tipo"] == "cuadro"]
print("Cuadros distintos (ID único):", cuadros["cuadro_id"].nunique())
multipagina = cuadros[(cuadros["orden_pagina"] == 1) & (cuadros["paginas_del_cuadro"] > 1)]
print("Bloques de cuadro multipágina:", len(multipagina))

primeras = cuadros[cuadros["orden_pagina"] == 1]
por_capitulo = primeras.groupby("capitulo").agg(bloques=("cuadro_id", "size"), paginas=("paginas_del_cuadro", "sum")).reset_index()
por_capitulo["capitulo"] = por_capitulo["capitulo"].astype(int)
por_capitulo = por_capitulo.sort_values("capitulo")
por_capitulo

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
conteo_tipo = cat["tipo"].value_counts()
ejes[0].bar(conteo_tipo.index, conteo_tipo.values, color="tab:blue")
ejes[0].set_title("Páginas por tipo")
ejes[1].bar(por_capitulo["capitulo"].astype(str), por_capitulo["paginas"], color="tab:green")
ejes[1].set_title("Páginas de cuadros por capítulo")
ejes[1].set_xlabel("capítulo")
plt.tight_layout()
plt.show()

## Familias que se extraen en el paso 03 (9.1, 4.1, 13.1, 14.1)

In [ ]:
interes = cuadros[cuadros["cuadro_id"].str.match(r"^(9\.1\.|4\.1\.|13\.1\.|14\.1\.)") & (cuadros["orden_pagina"] == 1)]
interes[["cuadro_id", "rango_paginas", "variante_etiqueta", "titulo"]]

## Páginas con texto pero sin etiqueta

In [ ]:
sin = cat[cat["tipo"] == "sin_etiqueta"]
print("Total:", len(sin))
sin[["pagina_pdf", "n_lineas", "primeras_lineas"]].head(12)